# Introduction to ACL2: Building Trustworthy Systems with Formal Verification

**What is formal verification?** In testing, you check that a program works on *some* inputs.
In formal verification, you *prove* it works on *all* inputs — forever.
No edge cases, no missed corner cases, no "works on my machine."

**ACL2** (A Computational Logic for Applicative Common Lisp) is an industrial-strength
theorem prover used to verify real systems: AMD processor floating-point units, the Java
Virtual Machine, commercial microprocessors, and cryptographic algorithms. It won the
2005 ACM Software System Award.

This notebook is a hands-on introduction. Every code cell is live ACL2 — modify and
re-run any of them.

## 1. ACL2 as a Calculator

ACL2 is based on Lisp, so expressions use prefix notation: `(+ 2 3)` instead of `2 + 3`.
Let's start with simple arithmetic.

In [1]:
; Basic arithmetic
(+ 2 3)

5

In [2]:
; Nesting works naturally — compute (3 * 4) + (10 / 2)
(+ (* 3 4) (/ 10 2))

17

ACL2 has booleans (`t` for true, `nil` for false), symbols, strings, and lists.
Lists are the fundamental data structure — they're built from pairs using `cons`.

In [3]:
; Build a list: '(1 2 3) is shorthand for (cons 1 (cons 2 (cons 3 nil)))
(list 1 2 3)

(1 2 3)

In [4]:
; car = first element, cdr = rest of the list
(car '(a b c))

a

In [5]:
(cdr '(a b c))

(b c)

## 2. Defining Functions

In ACL2, you define functions with `defun`. All functions must be **total** (defined on all
inputs) and **terminating** (no infinite loops). ACL2 checks this automatically — if it
can't prove your function terminates, it rejects it. This is the first piece of the
trustworthiness puzzle: *every function you write is guaranteed to terminate*.

In [6]:
; A function to compute the length of a list.
; ACL2 automatically proves this terminates because (cdr x) is smaller than x.
(defun my-len (x)
  (if (endp x)            ; endp tests for end-of-list (empty or not a cons)
      0
      (+ 1 (my-len (cdr x)))))


The admission of MY-LEN is trivial, using the relation O< (which is
known to be well-founded on the domain recognized by O-P) and the measure
(ACL2-COUNT X).  We observe that the type of MY-LEN is described by
the theorem (AND (INTEGERP (MY-LEN X)) (<= 0 (MY-LEN X))).  We used
primitive type reasoning.

Summary
Form:  ( DEFUN MY-LEN ...)
Rules: ((:FAKE-RUNE-FOR-TYPE-SET NIL))


my-len

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [7]:
; Test it
(my-len '(a b c d e))

5

In [8]:
; Append two lists together
(defun my-app (x y)
  (if (endp x)
      y
      (cons (car x)
            (my-app (cdr x) y))))


The admission of MY-APP is trivial, using the relation O< (which is
known to be well-founded on the domain recognized by O-P) and the measure
(ACL2-COUNT X).  We observe that the type of MY-APP is described by
the theorem (OR (CONSP (MY-APP X Y)) (EQUAL (MY-APP X Y) Y)).  We used
primitive type reasoning.

Summary
Form:  ( DEFUN MY-APP ...)
Rules: ((:FAKE-RUNE-FOR-TYPE-SET NIL))


my-app

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [9]:
(my-app '(1 2 3) '(4 5 6))

(1 2 3 4 5 6)

## 3. Proving Theorems — The Heart of Formal Verification

Here's where ACL2 diverges from ordinary programming. With testing, we might check:
- `(my-app '(1 2) '(3))` → `(1 2 3)` ✔
- `(my-app nil '(a))` → `(a)` ✔

But there are infinitely many inputs. How do we know it *always* works?

`defthm` lets us state a **universal** property and ACL2 will try to **prove** it
for all possible inputs using induction, rewriting, and other automated strategies.

### Theorem 1: Appending nil does nothing

In [10]:
; Prove: appending nil to any list returns that list (modulo true-list-fix).
; ACL2 discovers this needs induction on x and proves it automatically.
(defthm my-app-nil
  (equal (my-app x nil)
         (true-list-fix x)))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  These merge into one derived induction scheme.

We will induct according to a scheme suggested by (TRUE-LIST-FIX X),
while accommodating (MY-APP X NIL).

These suggestions were produced using the :induction rules MY-APP and
TRUE-LIST-FIX.  If we let (:P X) denote *1 above then the induction
scheme we'll use is
(AND (IMPLIES (NOT (CONSP X)) (:P X))
     (IMPLIES (AND (CONSP X) (:P (CDR X)))
              (:P X))).
This induction is justified by the same argument used to admit TRUE-LIST-FIX.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.

Subgoal *1/2
(IMPLIES (NOT (CONSP X))
         (EQUAL (MY-APP X NIL)
                (TRUE-LIST-FIX X))).

But simplification reduces this to T, using the :definitions MY-APP
and TRUE-LIST-FIX and the :executable-counterpart of EQUAL.

Subgoal *1/1
(IMPLIES (AND (CONSP X)
       

my-app-nil

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  155


### Theorem 2: Append is associative

This is a fundamental property: `(a ++ b) ++ c = a ++ (b ++ c)`. In conventional
programming, you'd have to *hope* this is true. In ACL2, we *prove* it.

In [11]:
; Prove associativity of append — for ALL lists x, y, z
(defthm my-app-assoc
  (equal (my-app (my-app x y) z)
         (my-app x (my-app y z))))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Three induction schemes are
suggested by this conjecture.  Subsumption reduces that number to two.
However, one of these is flawed and so we are left with one viable
candidate.  

We will induct according to a scheme suggested by (MY-APP X Y), while
accommodating (MY-APP X (MY-APP Y Z)).

These suggestions were produced using the :induction rule MY-APP. 
If we let (:P X Y Z) denote *1 above then the induction scheme we'll
use is
(AND (IMPLIES (AND (NOT (ENDP X)) (:P (CDR X) Y Z))
              (:P X Y Z))
     (IMPLIES (ENDP X) (:P X Y Z))).
This induction is justified by the same argument used to admit MY-APP.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.

Subgoal *1/2
(IMPLIES (AND (NOT (ENDP X))
              (EQUAL (MY-APP (MY-APP (CDR X) Y) Z)
                     (MY-APP (CDR X) (MY-APP Y Z))))
         (EQUAL (MY-APP (MY-APP X Y) Z)
                (MY-APP X (

my-app-assoc

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  409


### Theorem 3: Length distributes over append

If you append two lists, the length of the result equals the sum of their lengths.
Two things to notice:
1. ACL2 proves this *automatically* — you state the property, it finds the proof.
2. This holds for **every** pair of lists, not just the ones you tested.

In [12]:
; Prove: length of (append x y) = length(x) + length(y)
(defthm my-len-of-my-app
  (equal (my-len (my-app x y))
         (+ (my-len x) (my-len y))))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Three induction schemes are
suggested by this conjecture.  Subsumption reduces that number to two.
However, one of these is flawed and so we are left with one viable
candidate.  

We will induct according to a scheme suggested by (MY-APP X Y), while
accommodating (MY-LEN X).

These suggestions were produced using the :induction rules MY-APP and
MY-LEN.  If we let (:P X Y) denote *1 above then the induction scheme
we'll use is
(AND (IMPLIES (AND (NOT (ENDP X)) (:P (CDR X) Y))
              (:P X Y))
     (IMPLIES (ENDP X) (:P X Y))).
This induction is justified by the same argument used to admit MY-APP.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.

Subgoal *1/2
(IMPLIES (AND (NOT (ENDP X))
              (EQUAL (MY-LEN (MY-APP (CDR X) Y))
                     (+ (MY-LEN (CDR X)) (MY-LEN Y))))
         (EQUAL (MY-LEN (MY-APP X Y))
                (+ (MY-LEN X) (MY-LEN 

my-len-of-my-app

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)
Prover steps counted:  390


## 4. A Realistic Example: List Reversal

Reversing a list seems simple, but bugs in list manipulation have caused real security
vulnerabilities. Let's define reverse, prove it correct, and see how ACL2 catches
mistakes.

First, define reverse using an accumulator for efficiency:

In [13]:
; Reverse a list using a tail-recursive helper with an accumulator
(defun my-rev-aux (x acc)
  (if (endp x)
      acc
      (my-rev-aux (cdr x) (cons (car x) acc))))

(defun my-rev (x)
  (my-rev-aux x nil))


The admission of MY-REV-AUX is trivial, using the relation O< (which
is known to be well-founded on the domain recognized by O-P) and the
measure (ACL2-COUNT X).  We observe that the type of MY-REV-AUX is
described by the theorem 
(OR (CONSP (MY-REV-AUX X ACC)) (EQUAL (MY-REV-AUX X ACC) ACC)).  We
used primitive type reasoning.

Summary
Form:  ( DEFUN MY-REV-AUX ...)
Rules: ((:FAKE-RUNE-FOR-TYPE-SET NIL))


my-rev-aux

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)

Since MY-REV is non-recursive, its admission is trivial.  We observe
that the type of MY-REV is described by the theorem 
(OR (CONSP (MY-REV X)) (EQUAL (MY-REV X) NIL)).  We used the :type-
prescription rule MY-REV-AUX.

Summary
Form:  ( DEFUN MY-REV ...)
Rules: ((:TYPE-PRESCRIPTION MY-REV-AUX))


my-rev

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [14]:
; Test it
(my-rev '(1 2 3 4 5))

(5 4 3 2 1)

Now let's prove properties. First, a helper lemma about the accumulator version,
then the big result: **reversing a list preserves its length**.

In [15]:
; Lemma: the accumulator version appends acc in reverse
; This is the key insight ACL2 needs to reason about the tail-recursive version.
(defthm my-rev-aux-is-append
  (equal (my-rev-aux x acc)
         (my-app (my-rev-aux x nil) acc)))


Name the formula above *1.

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  Subsumption reduces that number to one.  

We will induct according to a scheme suggested by (MY-REV-AUX X ACC),
while accommodating (MY-REV-AUX X NIL).

These suggestions were produced using the :induction rule MY-REV-AUX.
If we let (:P ACC X) denote *1 above then the induction scheme we'll
use is
(AND (IMPLIES (AND (NOT (ENDP X))
                   (:P (CONS (CAR X) ACC) (CDR X)))
              (:P ACC X))
     (IMPLIES (ENDP X) (:P ACC X))).
This induction is justified by the same argument used to admit MY-REV-AUX.
Note, however, that the unmeasured variable ACC is being instantiated.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.

Subgoal *1/2
(IMPLIES (AND (NOT (ENDP X))
              (EQUAL (MY-REV-AUX (CDR X) (CONS (CAR X) ACC))
                     (MY-APP (MY-REV-AUX (CDR X) NIL)
                       

In [16]:
; THE BIG THEOREM: reversing a list preserves its length.
; This is true for ALL lists — not just the ones we tested.
(defthm my-len-of-my-rev
  (equal (my-len (my-rev x))
         (my-len x)))


ACL2 Warning [Non-rec] in ( DEFTHM MY-LEN-OF-MY-REV ...):  A :REWRITE
rule generated from MY-LEN-OF-MY-REV will be triggered only by terms
containing the function symbol MY-REV, which has a non-recursive definition.
Unless this definition is disabled, this rule is unlikely ever to be
used.


By the simple :definition MY-REV we reduce the conjecture to

Goal'
(EQUAL (MY-LEN (MY-REV-AUX X NIL))
       (MY-LEN X)).

Name the formula above *1.

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  Subsumption reduces that number to one.  

We will induct according to a scheme suggested by (MY-LEN X), while
accommodating (MY-REV-AUX X NIL).

These suggestions were produced using the :induction rules MY-LEN and
MY-REV-AUX.  If we let (:P X) denote *1 above then the induction scheme
we'll use is
(AND (IMPLIES (AND (NOT (ENDP X)) (:P (CDR X)))
              (:P X))
     (IMPLIES (ENDP X) (:P X))).
This induction is justified by the same argument used 

## 5. When Proofs Fail — ACL2 as a Bug Finder

What if we state something *false*? ACL2 won't just say "I can't prove it" — it will
search for a **counterexample** to show you exactly why the theorem is wrong.

Let's try to claim that reversing a list gives back the same list:

In [17]:
; This is FALSE — reverse doesn't return the same list!
; ACL2 will show us a counterexample. (Use thm for one-off proof attempts.)
(thm (equal (my-rev x) x))


By the simple :definition MY-REV we reduce the conjecture to

Goal'
(EQUAL (MY-REV-AUX X NIL) X).

Name the formula above *1.

Perhaps we can prove *1 by induction.  One induction scheme is suggested
by this conjecture.  

We will induct according to a scheme suggested by (MY-REV-AUX X NIL).

This suggestion was produced using the :induction rule MY-REV-AUX.
If we let (:P X) denote *1 above then the induction scheme we'll use
is
(AND (IMPLIES (AND (NOT (ENDP X)) (:P (CDR X)))
              (:P X))
     (IMPLIES (ENDP X) (:P X))).
This induction is justified by the same argument used to admit MY-REV-AUX.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.

Subgoal *1/2
(IMPLIES (AND (NOT (ENDP X))
              (EQUAL (MY-REV-AUX (CDR X) NIL)
                     (CDR X)))
         (EQUAL (MY-REV-AUX X NIL) X)).

By the simple :definition ENDP we reduce the conjecture to

Subgoal *1/2'
(IMPLIES (AND (CONSP X)
              (EQUAL (MY-REV-A

## 6. Guards — Connecting Logic to Efficient Execution

ACL2 functions are defined in *logic mode* where they work on all inputs.
But real systems need efficient, type-safe code. **Guards** bridge this gap:
they specify preconditions, and once verified, ACL2 can execute the function
more efficiently (skipping runtime checks).

This is how ACL2 verifications connect to real hardware and software.

In [18]:
; A guarded function: factorial, only defined for natural numbers.
; The :guard tells ACL2 the precondition for efficient execution.
(defun my-fact (n)
  (declare (xargs :guard (natp n)))
  (if (zp n)
      1
      (* n (my-fact (- n 1)))))


The admission of MY-FACT is trivial, using the relation O< (which is
known to be well-founded on the domain recognized by O-P) and the measure
(ACL2-COUNT N).  We observe that the type of MY-FACT is described by
the theorem (AND (INTEGERP (MY-FACT N)) (< 0 (MY-FACT N))).  We used
the :compound-recognizer rule ZP-COMPOUND-RECOGNIZER and primitive
type reasoning.

Computing the guard conjecture for MY-FACT....

The guard conjecture for MY-FACT is trivial to prove, given the :compound-
recognizer rules NATP-COMPOUND-RECOGNIZER and ZP-COMPOUND-RECOGNIZER,
primitive type reasoning and the :type-prescription rule MY-FACT. 
MY-FACT is compliant with Common Lisp.

Summary
Form:  ( DEFUN MY-FACT ...)
Rules: ((:COMPOUND-RECOGNIZER NATP-COMPOUND-RECOGNIZER)
        (:COMPOUND-RECOGNIZER ZP-COMPOUND-RECOGNIZER)
        (:FAKE-RUNE-FOR-TYPE-SET NIL)
        (:TYPE-PRESCRIPTION MY-FACT))


my-fact

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [19]:
; Verify guards: ACL2 proves that if n is a natural number,
; all recursive calls also satisfy the guard.
(verify-guards my-fact)


The event ( VERIFY-GUARDS MY-FACT) is redundant.  See :DOC redundant-
events.

Summary
Form:  ( VERIFY-GUARDS MY-FACT)
Rules: NIL


:redundant

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


In [20]:
; Now we can use it with confidence
(my-fact 10)

3628800

In [21]:
; Prove: factorial always returns a positive integer
(defthm my-fact-positive
  (implies (natp n)
           (posp (my-fact n))))


ACL2 Warning [Non-rec] in ( DEFTHM MY-FACT-POSITIVE ...):  A :REWRITE
rule generated from MY-FACT-POSITIVE will be triggered only by terms
containing the function symbol POSP, which has a non-recursive definition.
Unless this definition is disabled, this rule is unlikely ever to be
used.


But we reduce the conjecture to T, by the :compound-recognizer rule
POSP-COMPOUND-RECOGNIZER and the :type-prescription rule MY-FACT.

Q.E.D.

The storage of MY-FACT-POSITIVE depends upon the :compound-recognizer
rule POSP-COMPOUND-RECOGNIZER and the :type-prescription rule MY-FACT.

Summary
Form:  ( DEFTHM MY-FACT-POSITIVE ...)
Rules: ((:COMPOUND-RECOGNIZER POSP-COMPOUND-RECOGNIZER)
        (:TYPE-PRESCRIPTION MY-FACT))
Warnings:  Non-rec


my-fact-positive

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)


## 7. Why This Matters: Trustworthy Systems

What we've seen — termination checking, automated theorem proving, counterexample
generation, guard verification — scales to real systems:

| Domain | What ACL2 Verified |
|--------|-------------------|
| **Processors** | AMD floating-point division (after the Pentium FDIV bug) |
| **Security** | Java bytecode verifier, cryptographic algorithms |
| **Compilers** | Code generators proven to preserve semantics |
| **Operating Systems** | seL4 components, software fault isolation |

The key insight: **every theorem ACL2 proves is machine-checked**. Unlike testing,
which samples behavior, or code review, which depends on human attention, a formal
proof covers *all* cases. Once ACL2 says `Q.E.D.`, the property holds forever.

### The Verification Workflow

1. **Model** your system as ACL2 functions
2. **State** the properties you need (safety, correctness, security)
3. **Prove** them — ACL2 does most of the work; you supply key lemmas
4. **Execute** — guard-verified functions run efficiently as real code

### Going Further

- The [ACL2 community books](https://www.cs.utexas.edu/users/moore/acl2/) contain
  400,000+ verified theorems and functions — a massive library of trusted components.
- Use the `acl2-kg-mcp` Knowledge Graph to search and explore the entire library
  semantically.
- The [ACL2 documentation](https://www.cs.utexas.edu/users/moore/acl2/manuals/latest/)
  is comprehensive and includes hundreds of worked examples.

*All proofs in this notebook were checked automatically by ACL2. No proof steps were
skipped or paraphrased — what you see is what the prover produced.*